#  Building a RAG System with LangChain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document


#vectore store 
from langchain_community.vectorstores import Chroma

#utility 

import numpy as np
from typing import List

In [4]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



## sample data

In [5]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [14]:
# Create the data directory if it doesn't exist
os.makedirs("data", exist_ok=True)

# Save sample documents to files
for i, doc in enumerate(sample_docs):
    with open(f"data/doc_{i}.txt", "w", encoding="utf-8") as f:
        f.write(doc)

print("Sample documents created successfully!")


Sample documents created successfully!


# document loading

In [15]:
from langchain_community.document_loaders import DirectoryLoader


loader = DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'},
)


documents = loader.load()


print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200] + "...")

Loaded 3 documents

First document preview:

    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. Ther...


In [16]:
documents

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    '),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natur

# documents splitting 

In [17]:
#initialize text splitter 

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]  # Hierarchy of separators
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 5 chunks from 3 documents

Chunk example:
Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experie...
Metadata: {'source': 'data\\doc_0.txt'}


In [18]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of in

# embedding model

In [22]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


In [23]:
sample_text = "i am ml enginner"
embeddings = OpenAIEmbeddings()
embeddings


OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x00000194716EE900>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x00000194716EF8C0>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [25]:
vector = embeddings.embed_query(sample_text)
vector

[-0.02594597637653351,
 -0.005585878621786833,
 -0.019786041229963303,
 -0.03024968132376671,
 -0.009356776252388954,
 0.01638982817530632,
 -0.014079852029681206,
 0.007452421355992556,
 -0.008841156959533691,
 -0.023691000416874886,
 0.01002364419400692,
 0.027403460815548897,
 -0.0026743467897176743,
 0.010958634316921234,
 -0.002887469483539462,
 -0.010587388649582863,
 0.010773011483252048,
 0.0016431077383458614,
 -0.00320371612906456,
 -0.0021277901250869036,
 -0.006015561521053314,
 0.030222181230783463,
 0.023058507591485977,
 -0.037922099232673645,
 -0.004179955925792456,
 -0.00897178053855896,
 0.0027912205550819635,
 -0.0303046815097332,
 -0.009831146337091923,
 -0.0013079550117254257,
 0.010601138696074486,
 -0.004269329831004143,
 -0.015138590708374977,
 -0.019896039739251137,
 -0.019208546727895737,
 -0.019744791090488434,
 0.011914249509572983,
 -0.00480557419359684,
 0.01927729696035385,
 0.0019765417091548443,
 0.029782185330986977,
 0.015606085769832134,
 0.003664336

## Intilialize the ChromaDB Vector Store And Stores the chunks in Vector Representation

In [26]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of in

In [27]:
## Create a Chromdb vector store
persist_directory="./chroma_db"

## Initialize Chromadb with Open AI embeddings
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(),
    persist_directory=persist_directory,
    collection_name="rag_collection"

)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

Vector store created with 5 vectors
Persisted to: ./chroma_db


# Test similarity search 

In [30]:
query = "what is machine learning"
similar_docs = vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (

In [31]:
query = "what is NLp"
similar_docs = vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are part

In [32]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: what is NLp

Top 3 similar chunks:

--- Chunk 1 ---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recogn...
Source: data\doc_2.txt

--- Chunk 2 ---
Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of i...
Source: data\doc_1.txt

--- Chunk 3 ---
Neural Networks (RNNs) and Transformers 
    excel at sequential data processing....
Source: data\doc_1.txt


##  Advanced Similarity Search With Scores

In [33]:
results_scores=vectorstore.similarity_search_with_score(query,k=3)
results_scores

[(Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
  0.5382189154624939),
 (Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

# LCEL (LangChain Expression Language)



In [34]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model = "gpt-4o-mini")

In [35]:
test_responses = llm.invoke("what is ML")
test_responses

AIMessage(content="ML stands for Machine Learning, a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit programming. Instead of being programmed with specific rules, machine learning algorithms learn from data, identifying patterns and making decisions based on that data.\n\nThere are several types of machine learning:\n\n1. **Supervised Learning**: The model is trained on a labeled dataset, meaning the input data is paired with the correct output. The goal is to learn a function that maps inputs to the desired outputs. Common applications include classification and regression tasks.\n\n2. **Unsupervised Learning**: The model is trained on data without labeled responses. It aims to identify patterns, groupings, or structures in the data. Examples include clustering and dimensionality reduction.\n\n3. **Semi-supervised Learning**: This approach combines both labeled and unlab

In [36]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [37]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
You are a helpful AI assistant.

Answer the question only using the context below.

Context:
{context}

Question:
{question}

Answer:
"""
)

In [38]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [39]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [40]:
response = rag_chain.invoke(
    "What is machine learning?"
)

print(response)

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.


## Conversational Rag using LCEL manually 

In [41]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

In [42]:
# prompt with chat history 

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         """You are a helpful AI assistant.

Use the given context to answer the user's question.
If the answer is not in the context, say you don't know.

Context:
{context}
"""),

        MessagesPlaceholder(variable_name="chat_history"),

        ("human", "{question}")
    ]
)

In [43]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [48]:
#Create the LCEL Chain

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

rag_chain = (
    {
        "context": (lambda x: x["question"]) | retriever | format_docs,
        "question": lambda x: x["question"],
        "chat_history": lambda x: x["chat_history"],
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [49]:
chat_history = []

In [50]:
question = "What is machine learning?"

response = rag_chain.invoke(
    {
        "question": question,
        "chat_history": chat_history
    }
)

print(response)

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.


In [51]:
chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=response)
    ]
)

In [52]:
question = "What are its applications?"

response = rag_chain.invoke(
    {
        "question": question,
        "chat_history": chat_history
    }
)

print(response)

The context does not explicitly list the applications of machine learning. However, based on the information provided, machine learning, particularly deep learning, has applications in fields such as:

- Natural Language Processing (NLP), including tasks like text classification, named entity recognition, sentiment analysis, machine translation, and question answering.
- Computer vision, using Convolutional Neural Networks (CNNs) for image processing.
- Speech recognition.

These are some key areas where machine learning is applied.


# Conversational Memory using Lcel RunnableWithMessageHistory , InMemoryChatMessageHistory

In [53]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [54]:
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]

In [55]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a helpful assistant.

Use the context below to answer.

Context:
{context}
"""
        ),

        MessagesPlaceholder(variable_name="chat_history"),

        ("human", "{question}")
    ]
)

In [56]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

get_question = RunnableLambda(lambda x: x["question"])

chain = (
    {
        "context": get_question | retriever | format_docs,
        "question": get_question,
        "chat_history": RunnableLambda(lambda x: x["chat_history"]),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [57]:
with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

In [58]:
response = with_history.invoke(
    {
        "question": "What is machine learning?"
    },
    config={
        "configurable": {
            "session_id": "user_1"
        }
    }
)

print(response)

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It involves training models to recognize patterns and make decisions based on data. There are three main types of machine learning: supervised learning, which uses labeled data; unsupervised learning, which finds patterns in unlabeled data; and reinforcement learning, which learns through interaction with the environment.


In [59]:
response = with_history.invoke(
    {
        "question": "What are its applications?"
    },
    config={
        "configurable": {
            "session_id": "user_1"
        }
    }
)

print(response)

Machine learning has a wide range of applications across various fields, including:

1. **Natural Language Processing (NLP):** Tasks such as text classification, named entity recognition, sentiment analysis, machine translation, and question answering. Modern NLP heavily relies on transformer architectures like BERT, GPT, and T5.

2. **Computer Vision:** Image recognition, object detection, facial recognition, and image segmentation, often using Convolutional Neural Networks (CNNs).

3. **Speech Recognition:** Converting spoken language into text and understanding speech commands.

4. **Recommendation Systems:** Suggesting products, movies, or content based on user preferences and behavior.

5. **Healthcare:** Disease diagnosis, medical image analysis, and personalized treatment recommendations.

6. **Finance:** Fraud detection, algorithmic trading, and credit scoring.

7. **Autonomous Vehicles:** Perception, decision-making, and control systems for self-driving cars.

8. **Robotics:**